## 3.1 Naive Self-Attention

In Chapter 2 we produced `input_embeddings` of shape `[batch, seq_len, d_model]` — 
one vector per token. The problem: every token vector is independent. The model has 
no way to relate "dog" to "bites" in "dog bites man".

Self-attention fixes this by producing a **context vector** for each token — a weighted 
mix of all other token embeddings, where the weights reflect how relevant each token is.

### The idea
For a query token `q` and every other token `x_i`, compute a score:

    score_i = dot(q, x_i)

Convert scores to weights via softmax (so they sum to 1), then take the weighted sum 
of all token vectors. The result is a context vector that blends information from the 
whole sequence.

Here we do this manually with loops for token at position 1.

## 3.2 Vectorized Self-Attention

The loop approach works but is slow. We can compute attention for all tokens 
simultaneously using matrix multiplication.

`inputs @ inputs.T` gives us a `[seq_len, seq_len]` matrix where entry `[i, j]` 
is the dot product between token `i` and token `j` — the attention score of token 
`i` attending to token `j`.

Softmax with `dim=-1` normalizes each **row** independently, so each token gets 
its own probability distribution over all other tokens.

The final matmul `attn_weights @ inputs` computes the weighted sum for all tokens 
at once, giving context vectors of shape `[seq_len, d_model]`.

We verify row 1 matches the loop result from 3.1.

In [21]:
# All attention scores at once: [seq_len, seq_len]
attn_scores = inputs @ inputs.T

# Normalize each row independently
attn_weights = torch.softmax(attn_scores, dim=-1)

# Weighted sum: [seq_len, d_model]
context_vecs = attn_weights @ inputs

print("attn_weights shape:", attn_weights.shape)# Compute all attention scores at once via matrix multiply
# entry [i,j] = dot(token_i, token_j)
attn_scores = inputs @ inputs.T  # [seq_len, seq_len]

# Normalize each row into a probability distribution
# dim=-1 means normalize along the last axis (across columns per row)
attn_weights = torch.softmax(attn_scores, dim=-1)  # [seq_len, seq_len]

# Weighted sum for all tokens at once
# each row i of attn_weights blends all token vectors into one context vector
context_vecs = attn_weights @ inputs  # [seq_len, d_model]

print("attn_weights shape:", attn_weights.shape)
print("context_vecs shape:", context_vecs.shape)

# Sanity check — row 1 should match our manual loop result from 3.1
print("row 1 matches previous:", torch.allclose(context_vecs[1], context_vec))
print("context_vecs shape:", context_vecs.shape)
print("row 1 matches previous:", torch.allclose(context_vecs[1], context_vec))

attn_weights shape: torch.Size([6, 6])
attn_weights shape: torch.Size([6, 6])
context_vecs shape: torch.Size([6, 3])
row 1 matches previous: True
context_vecs shape: torch.Size([6, 3])
row 1 matches previous: True


## 3.3 Self-Attention with Learned Projections

The previous approach used raw token vectors as queries and keys. The problem: 
the same vectors are used for both "what am I looking for" and "what do I contain" 
— there's no flexibility.

Real self-attention introduces three learned weight matrices that project inputs 
into separate spaces:

- **W_q** — projects input into query space ("what am I looking for?")
- **W_k** — projects input into key space ("what do I contain?")  
- **W_v** — projects input into value space ("what do I pass forward?")

The formula:

    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

The `sqrt(d_k)` scaling prevents dot products from getting too large as 
dimensionality grows, which would push softmax into regions with tiny gradients.

Note: output is now `[seq_len, d_k]` not `[seq_len, d_in]` — Q, K, V live 
in their own projected space which can have different dimensionality to the input.

In [22]:
torch.manual_seed(123)

d_in = 3   # input embedding dimension (from Ch2)
d_k = 2    # dimension of Q, K, V projections (hyperparameter, can differ from d_in)

# Learned projection matrices — wrapped in nn.Parameter so PyTorch
# tracks them and updates them during training via backprop
W_q = torch.nn.Parameter(torch.rand(d_in, d_k))
W_k = torch.nn.Parameter(torch.rand(d_in, d_k))
W_v = torch.nn.Parameter(torch.rand(d_in, d_k))

# Project inputs into Q, K, V spaces
# each token vector [d_in] gets mapped to a new vector [d_k]
Q = inputs @ W_q  # [seq_len, d_k]
K = inputs @ W_k  # [seq_len, d_k]
V = inputs @ W_v  # [seq_len, d_k]

# Scaled dot-product attention
# divide by sqrt(d_k) to keep gradients stable as d_k grows
attn_scores = Q @ K.T / d_k**0.5   # [seq_len, seq_len]
attn_weights = torch.softmax(attn_scores, dim=-1)
context_vecs = attn_weights @ V     # [seq_len, d_k]

print("Q shape:", Q.shape)
print("attn_weights shape:", attn_weights.shape)
print("context_vecs shape:", context_vecs.shape)  # [6, 2] — projected space

Q shape: torch.Size([6, 2])
attn_weights shape: torch.Size([6, 6])
context_vecs shape: torch.Size([6, 2])


## 3.4 SelfAttentionV1 — Wrapping into a Class

We now package the attention logic into a reusable `nn.Module` class.

`nn.Module` is PyTorch's base class for all neural network layers. Subclassing it gives us:

- **Parameter tracking** — anything wrapped in `nn.Parameter` is automatically 
  registered. When you later call `optimizer.step()`, PyTorch knows which tensors 
  to update via backprop.
- **`forward()`** — defines the computation. Called automatically when you do `layer(x)`.

`nn.Parameter` vs plain `torch.rand`:
- `torch.rand` — just a tensor, PyTorch ignores it during training
- `nn.Parameter` — PyTorch tracks it, computes gradients, updates it during training

The weights start random. Training will adjust them so attention learns to focus 
on useful token relationships. We will cover training in Chapter 5.

Notice `grad_fn=<MmBackward0>` in the output — PyTorch has attached a gradient 
function to the result, ready for backprop.

In [23]:
import torch
import torch.nn as nn

class SelfAttentionV1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()  # required — initializes nn.Module internals

        # Learned projection matrices registered as parameters
        # PyTorch will update these during training via backprop
        self.W_q = nn.Parameter(torch.rand(d_in, d_out))
        self.W_k = nn.Parameter(torch.rand(d_in, d_out))
        self.W_v = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # Project inputs into Q, K, V spaces
        Q = x @ self.W_q
        K = x @ self.W_k
        V = x @ self.W_v

        # Scaled dot-product attention
        attn_scores = Q @ K.T / K.shape[-1]**0.5
        attn_weights = torch.softmax(attn_scores, dim=-1)

        # Weighted sum of value vectors
        return attn_weights @ V

torch.manual_seed(123)
sa = SelfAttentionV1(d_in=3, d_out=2)
print(sa(inputs))  # grad_fn confirms gradients are tracked

tensor([[0.1671, 0.6265],
        [0.1702, 0.6414],
        [0.1636, 0.6090],
        [0.1705, 0.6434],
        [0.1647, 0.6144],
        [0.1689, 0.6351]], grad_fn=<MmBackward0>)


## 3.5 SelfAttentionV2 — Replacing nn.Parameter with nn.Linear

V1 used `nn.Parameter(torch.rand(...))` which initializes weights from a uniform 
distribution — bad for training as it can cause unstable gradients early on.

V2 swaps to `nn.Linear(d_in, d_out, bias=False)` which uses smarter default 
initialization (Kaiming uniform). The call syntax also changes:

In [24]:
class SelfAttentionV2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        # nn.Linear is preferred over nn.Parameter —
        # better default weight initialization, cleaner syntax
        # bias=False follows standard attention implementations
        self.W_q = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        # nn.Linear handles the matmul internally: equivalent to x @ W.T
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        attn_scores = Q @ K.T / K.shape[-1]**0.5
        attn_weights = torch.softmax(attn_scores, dim=-1)
        return attn_weights @ V

torch.manual_seed(123)
sa = SelfAttentionV2(d_in=3, d_out=2)
print(sa(inputs))  # different values from V1 due to initialization, same structure

tensor([[-0.3799,  0.0564],
        [-0.3938,  0.0638],
        [-0.3806,  0.0574],
        [-0.3902,  0.0617],
        [-0.3810,  0.0575],
        [-0.3906,  0.0623]], grad_fn=<MmBackward0>)


## 3.6 Causal Masking

The attention we've built so far lets every token attend to every other token — 
including future tokens. This is wrong for a language model.

When predicting the next word, the model should only see past and current tokens, 
not future ones. This is called **causal** attention — the same concept as causal 
systems in signals and systems, where output at time `i` depends only on inputs 
up to time `i`.

We enforce this by masking the upper triangle of the `[seq_len, seq_len]` attention 
score matrix with `-inf` before softmax. Since `exp(-inf) = 0`, softmax converts 
these to exactly zero — those token pairs contribute nothing.

In [25]:
# Recompute attention scores using our V2 model
Q = sa.W_q(inputs)
K = sa.W_k(inputs)
attn_scores = Q @ K.T / K.shape[-1]**0.5  # [seq_len, seq_len]

# Build upper triangular mask — 1s where we want to block attention
# diagonal=1 means the main diagonal stays 0 (tokens can attend to themselves)
mask = torch.triu(torch.ones(6, 6), diagonal=1)

# Replace masked positions with -inf — softmax will convert these to 0
masked_scores = attn_scores.masked_fill(mask.bool(), float('-inf'))

# Softmax now only distributes weight over current and past tokens
attn_weights = torch.softmax(masked_scores, dim=-1)

# Upper triangle should be all zeros — causal constraint enforced
print(attn_weights.round(decimals=2))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4600, 0.5400, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3300, 0.3400, 0.3300, 0.0000, 0.0000, 0.0000],
        [0.2400, 0.2700, 0.2200, 0.2700, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000],
        [0.1600, 0.1800, 0.1500, 0.1800, 0.1500, 0.1700]],
       grad_fn=<RoundBackward1>)


## 3.7 Dropout on Attention Weights

Dropout is a regularization technique that randomly zeros out values during 
training and scales the remaining ones up to compensate.

With `p=0.5`, each attention weight has a 50% chance of being zeroed. The 
surviving weights are scaled by `1/(1-p) = 2` to keep the expected values 
consistent.

**Why** — forces the model not to rely too heavily on any single token. If the 
model knows any weight might get dropped randomly, it can't learn "token 3 is 
always the most important." It must distribute attention more robustly.

**Important** — dropout is only active during training. During inference it is 
automatically disabled and all weights are used as-is.

Notice rows no longer sum to 1 after dropout — some weights were zeroed out.

In [26]:
import torch.nn as nn

# p=0.5 means 50% of weights get zeroed randomly during training
# surviving weights are scaled by 1/(1-p)=2 to compensate
dropout = nn.Dropout(p=0.5)

attn_weights_dropped = dropout(attn_weights)

# zeros appear in two places:
# 1. upper triangle — from causal masking (deterministic)
# 2. lower triangle — from dropout (random)
# rows no longer sum to 1 because some weights were zeroed
print(attn_weights_dropped.round(decimals=2))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9200, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6500, 0.0000, 0.0000, 0.0000],
        [0.4800, 0.0000, 0.4500, 0.0000, 0.0000, 0.0000],
        [0.4000, 0.4100, 0.0000, 0.4100, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.3500]],
       grad_fn=<RoundBackward1>)


## 3.8 CausalAttention — Full Class

We now combine everything into a single production-ready `CausalAttention` class:
- Learned Q, K, V projections via `nn.Linear`
- Causal mask stored via `register_buffer`
- Dropout on attention weights
- Batched input support

Two important additions compared to V2:

**`register_buffer`** — stores the mask as part of the module but NOT as a 
learnable parameter. This means:
- It moves to GPU automatically when you call `model.cuda()`
- It gets saved when you call `torch.save(model)`
- It is never updated during training — it is a fixed constant

Using plain `self.mask = torch.triu(...)` would lose the mask on GPU transfer.

**`K.transpose(1, 2)`** — input is now batched: `[B, T, d_k]`. We can no longer 
use `K.T` which would flip all dimensions including batch. We only want to swap 
the last two dims to get `[B, d_k, T]` so the matmul works:

In [27]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)

        # register_buffer: mask is a constant, not a learnable parameter
        # moves to GPU automatically with the model, saved with torch.save()
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        B, T, d_in = x.shape  # unpack batch size, sequence length, embedding dim

        Q = self.W_q(x)  # [B, T, d_out]
        K = self.W_k(x)  # [B, T, d_out]
        V = self.W_v(x)  # [B, T, d_out]

        # transpose dims 1,2 only — preserves batch dim
        # [B, T, d_out] @ [B, d_out, T] = [B, T, T]
        attn_scores = Q @ K.transpose(1, 2) / K.shape[-1]**0.5

        # [:T, :T] handles sequences shorter than context_length
        attn_scores = attn_scores.masked_fill(self.mask.bool()[:T, :T], float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        return attn_weights @ V  # [B, T, d_out]

# test with a batch of 2 sequences
torch.manual_seed(123)
batch = torch.stack([inputs, inputs], dim=0)  # [2, 6, 3]
ca = CausalAttention(d_in=3, d_out=2, context_length=6, dropout=0.0)
print(ca(batch).shape)  # expect [2, 6, 2]

torch.Size([2, 6, 2])


## 3.9 MultiHeadAttention

A single attention head learns one way to relate tokens. Multi-head attention 
runs several attention heads in parallel, each learning different relationships:
- one head might learn syntax
- another semantics  
- another coreference

**How it works** — instead of one big attention operation in `d_out` dimensions, 
we split `d_out` across `num_heads` smaller heads, each operating in `head_dim = d_out // num_heads` dimensions. The results are concatenated and projected via `W_o`.

**Shape flow:**

In [28]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # dimension each head operates in

        # single projection for all heads — split into heads in forward()
        self.W_q = nn.Linear(d_in, d_out, bias=False)
        self.W_k = nn.Linear(d_in, d_out, bias=False)
        self.W_v = nn.Linear(d_in, d_out, bias=False)

        # output projection — mixes information across all heads
        self.W_o = nn.Linear(d_out, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        B, T, d_in = x.shape

        Q = self.W_q(x)  # [B, T, d_out]
        K = self.W_k(x)
        V = self.W_v(x)

        # split d_out into num_heads × head_dim
        # then transpose so heads are dim 1: [B, num_heads, T, head_dim]
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # attention within each head — transpose last two dims of K
        # [B, num_heads, T, head_dim] @ [B, num_heads, head_dim, T]
        attn_scores = Q @ K.transpose(2, 3) / self.head_dim**0.5

        # causal mask — each head sees only past and current tokens
        attn_scores = attn_scores.masked_fill(self.mask.bool()[:T, :T], float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # weighted sum per head: [B, num_heads, T, head_dim]
        out = attn_weights @ V

        # concatenate heads back: [B, num_heads, T, head_dim] -> [B, T, d_out]
        # contiguous() required before view() after transpose
        out = out.transpose(1, 2).contiguous().view(B, T, -1)

        # final projection mixes across heads
        return self.W_o(out)

torch.manual_seed(123)
mha = MultiHeadAttention(d_in=3, d_out=4, context_length=6, dropout=0.0, num_heads=2)
print(mha(batch).shape)  # expect [2, 6, 4]

torch.Size([2, 6, 4])
